In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv(r'F:\Hackathons\Kaggle-Nemotron\andy_dataset\top_per_category.csv')
len(df)

3000

In [3]:
# ── config ──
ROW_IDX = 0                  # which CSV row to inspect
USE_SYSTEM_PROMPT = 1        # 1 = include the small system prompt (set 0 to match the no-system 0.85 recipe)

SYSTEM_PROMPT = (
    "You solve a puzzle defined only by the examples in the prompt. "
    "Infer the exact rule from the examples, apply it to the query, and verify your "
    "result reproduces the examples before answering. Reason step by step, then output "
    "the final answer once as \\boxed{...} with nothing after it."
)
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

# tokenizer for rendering the REAL chat template:
#   Kaggle: a kagglehub model path, e.g. the nemotron model dir
#   Local : the HF id (needs internet) "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
TOKENIZER_PATH = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print("config ready. ROW_IDX =", ROW_IDX, "| USE_SYSTEM_PROMPT =", USE_SYSTEM_PROMPT)

config ready. ROW_IDX = 0 | USE_SYSTEM_PROMPT = 1


In [6]:
import re

# ── pick the row (FIX: df.iloc[ROW_IDX], not df[ROW_IDX]) ──
row     = df.iloc[ROW_IDX]
prompt  = str(row["prompt"])
answer  = str(row["answer"])
cot     = str(row["generated_cot"])

# clean any stray box from the cot, then rebuild the canonical assistant target
cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
user_content      = prompt + PROMPT_SUFFIX
assistant_content = "<think>\n" + cot_cleaned + "\n</think>\n\\boxed{" + answer + "}"

# message lists: train (with assistant) + infer (no assistant)
sys_msgs   = [{"role": "system", "content": SYSTEM_PROMPT}] if USE_SYSTEM_PROMPT else []
infer_msgs = sys_msgs + [{"role": "user", "content": user_content}]
train_msgs = infer_msgs + [{"role": "assistant", "content": assistant_content}]

# print("type   :", row.get("type", "?"))
# print("answer :", repr(answer))
print("\n===== USER content (problem + suffix) =====\n", user_content)
# print("\n===== ASSISTANT target (what SFT learns) — head =====\n", assistant_content[:300])
print("...\n===== ASSISTANT target — tail =====\n", assistant_content)


===== USER content (problem + suffix) =====
 In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
87$78 = 165
30$52 = 82
54|41 = 5441
Now, determine the result for: 95$62
Please put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`
...
===== ASSISTANT target — tail =====
 <think>
Step 1: Examine the example equations
We are given:
- 8778 = 165
- 3052 = 82

We are to determine what operation the symbol "" represents.

Step 2: Test possible operations for ""
Try addition:
- 87 + 78 = 165 ✓
- 30 + 52 = 82 ✓

Both examples match when "" is interpreted as addition.

Step 3: Verify the rule against the examples
For 8778 :
87 + 78 = (80+7) + (70+8) = 150 + 15 = 165

For 3052 :
30 + 52 = 30 + 50 + 2 = 80 + 2 = 82

The addition rule holds for both given examples.

Step 4: Apply the rule to 9562
Using the discovered rule:
95 + 62 = (90+5) + (60+2) = 150 + 7 = 157

Thus, 9562 = 157
</think>
\boxed{157}


In [5]:
# ── render the REAL chat template (what the model actually receives) ──
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(TOKENIZER_PATH, trust_remote_code=True)

def render(msgs, add_gen):
    try:
        return tok.apply_chat_template(msgs, tokenize=False,
                                       add_generation_prompt=add_gen, enable_thinking=True)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=add_gen)

infer_text = render(infer_msgs, True)    # what's fed at EVAL/generation time
train_text = render(train_msgs, False)   # the full SFT sequence

print("=" * 70)
print("INFERENCE RENDER  (add_generation_prompt=True)  — exactly what eval feeds the model")
print("=" * 70)
print(infer_text)
print("\nprompt tokens:", len(tok(infer_text, add_special_tokens=False)["input_ids"]))

print("\n" + "=" * 70)
print("FULL TRAINING RENDER  (system + user + assistant)")
print("=" * 70)
print(train_text[:1200], "\n...\n", train_text[-300:])
print("\ntotal training tokens:", len(tok(train_text, add_special_tokens=False)["input_ids"]))

f:\Hackathons\gemma4b-kaggle\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
# ── OPTIONAL: actually generate on this one prompt (needs `model` already loaded) ──
RUN_MODEL = 0   # set 1 if a Nemotron `model` is loaded in this kernel (e.g. on Kaggle)

if RUN_MODEL and "model" in dir():
    import torch
    enc = tok(infer_text, return_tensors="pt").to(model.device)
    plen = enc["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=2048, do_sample=False,
                             temperature=None, top_p=None, top_k=None,
                             pad_token_id=tok.pad_token_id)
    gen = tok.decode(out[0][plen:], skip_special_tokens=True)
    j = gen.rfind("\\boxed{"); pred = None
    if j != -1:
        d, k = 1, j + 7
        while k < len(gen) and d > 0:
            d += gen[k] == "{"; d -= gen[k] == "}"; k += 1
        pred = gen[j + 7:k - 1].strip()
    print("===== MODEL OUTPUT =====\n", gen[:1500])
    print("\npred :", repr(pred))
    print("gold :", repr(answer))
    print("MATCH:", str(pred).strip().lower() == answer.strip().lower())
else:
    print("RUN_MODEL=0 (or no `model` loaded) — template inspection only. "
          "Set RUN_MODEL=1 after loading the model to generate on this prompt.")